# Chapter 9: Watching the Money and the Machines

**Observability, cost, and the ROI case — a runnable learning walkthrough**

This notebook turns a synthetic maintenance-planning workflow into the record both engineering and finance need: every run carries caller-derived identity, a versioned price, a deterministic budget, and a business outcome. The default path is offline. It makes no model calls, needs no credential, and produces the same teaching results on every run.

The optional final section shows the pinned OpenAI Agents SDK wiring without calling a model. A live call happens only when you set both `OPENAI_API_KEY` and `CHAPTER9_RUN_LIVE=1` yourself.


## Start here

Run the cells from top to bottom once. The sequence mirrors the chapter:

1. Build Total Cost of Ownership (TCO), not a token-only estimate.
2. Read one agent run as a trace and expose a zero-error loop.
3. Normalize SDK usage fields and price every generation from a versioned card.
4. Carry identity and receipts across internal and counterparty boundaries.
5. Compute paired production metrics with bounded uncertainty.
6. Stop repeated or over-budget actions before they spend.
7. Route by measured step cost and the cost-quality frontier.
8. Generate finance and engineering views from the same ledger.
9. Export a hashed value-review evidence pack and, optionally, local OTLP traces.


## 0. Setup


In [ ]:
from __future__ import annotations

import json
import os
import sys
import tempfile
from pathlib import Path

chapter_root = Path.cwd()
if not (chapter_root / "chapter9").exists():
    candidate = chapter_root / "chapter 9"
    if candidate.exists():
        chapter_root = candidate
if str(chapter_root) not in sys.path:
    sys.path.insert(0, str(chapter_root))

from chapter9.border import call_counterparty
from chapter9.budget import LoopDetected, RunBudget
from chapter9.anomalies import credit_depletion_state, detector_coverage
from chapter9.dashboard import build_engineering_dashboard, build_finance_dashboard
from chapter9.evidence import emit_value_evidence, verify_manifest
from chapter9.live import build_maintenance_agent, make_live_context, run_live_case
from chapter9.maintenance import MaintenanceHarness, load_cases
from chapter9.metrics import cost_per_outcome_by_plant, production_metrics
from chapter9.pricing import load_price_card, normalize_usage, price_model_usage
from chapter9.replay import ActivityJournal
from chapter9.quality import JudgeCalibrationMonitor, calibration_report, stable_human_review_sample
from chapter9.routing import (
    choose_cheapest_acceptable,
    load_candidates,
    pareto_frontier,
    routing_change_gate,
    token_share_by_step,
)

cases = load_cases()
harness = MaintenanceHarness()
traces = harness.run_all(cases)
card = load_price_card()
print(f"Loaded {len(cases)} synthetic cases and emitted {len(traces)} traces.")


## 1. What the CFO actually wants to know

Token price is one line in TCO. The denominator matters just as much: all attempts cost money, while only business-accepted outcomes count as successes. The companion therefore defines cost per successful outcome as fully loaded cost across every attempt divided by approved work orders.

The synthetic review rate is `$0.98` per minute, matching the chapter's illustrative `$2.94` for a three-minute review. It is intentionally larger than the model line so an optimization targets the real cost rather than the most fashionable one.


In [ ]:
metrics = production_metrics(traces)
{
    "attempts": metrics["attempts"],
    "approved_outcomes": metrics["approved_outcomes"],
    "machine_cost_usd": metrics["machine_cost_usd"],
    "human_review_cost_usd": metrics["human_review_cost_usd"],
    "cost_per_successful_outcome_usd": metrics["cost_per_successful_outcome_usd"],
}


In [ ]:
[row.model_dump() for row in cost_per_outcome_by_plant(traces)]


Plant D is above the human baseline, while Plants A, B, and C are below their severity-specific baselines. Normalizing by severity avoids punishing a plant simply because it handles harder failures.


## 2. Tracing the machines: from spans to answers

A green request is not enough. The trace waterfall reveals what the agent did inside the request: generations, tools, retries, loops, reasoning, duration, and cost.

![A maintenance-agent trace waterfall](assets/figure_9_1_trace_viewer.png)


In [ ]:
sample = next(trace for trace in traces if trace.identity.plant == "Plant B")
[
    {
        "kind": span.kind,
        "name": span.name,
        "step": span.step_type,
        "duration_ms": span.duration_ms,
        "cost_usd": round(span.cost_usd, 5),
        "status": span.status,
        "reason": span.attributes.get("reason"),
    }
    for span in sample.spans
]


The denied `loop_breaker` span is a healthy control, not an application error. It proves the harness noticed an identical parts lookup and refused to buy it twice.


### A thin usage translation layer

The checked-in processor accepts flat teaching fields and the pinned SDK's nested `input_tokens_details` and `output_tokens_details`. Everything downstream reads the normalized shape. Provider or SDK naming changes therefore touch one adapter rather than every dashboard.


In [ ]:
sdk_shaped_usage = {
    "input_tokens": 10_000,
    "input_tokens_details": {"cached_tokens": 6_000, "cache_write_tokens": 1_000},
    "output_tokens": 2_000,
    "output_tokens_details": {"reasoning_tokens": 1_200},
}
normalized = normalize_usage(sdk_shaped_usage)
cost = price_model_usage(normalized, model="maintenance-mid", card=card)
{"normalized": normalized.model_dump(), "cost": cost.model_dump(), "total": cost.total}


The five cost fields suggest five different fixes: stabilize an uncached prefix, reuse cache writes, lower reasoning effort, constrain visible output, or route the step to a smaller model. An unexplained total cannot tell you which lever to pull.

![Trace propagation through the harness and across partner borders](assets/figure_9_2_trace_propagation.png)


### Counterparty receipts and replay without repeated effects


In [ ]:
from dataclasses import dataclass

@dataclass
class Reply:
    status: str
    task_id: str
    server_info: dict
    metadata: dict

class SyntheticSupplier:
    def __init__(self):
        self.calls = 0
    def send(self, task, *, headers):
        self.calls += 1
        return Reply(
            status="retry" if self.calls == 1 else "completed",
            task_id="supplier-task-42",
            server_info={"name": "synthetic-supplier", "version": "2"},
            metadata={"cost": 0.08, "receipt_id": "receipt-9"},
        )

supplier = SyntheticSupplier()
_, receipt = call_counterparty(
    supplier,
    {"part": "PART-BEARING-A"},
    tenant="heavy-things-manufacturing",
    case_id="case-9",
)
receipt


In [ ]:
journal = ActivityJournal()
journal.record(
    activity_id="tool-write-1",
    kind="tool",
    name="write_work_order",
    request={"asset_id": "A-PUMP-14"},
    response={"work_order": "WO-RECORDED-1"},
)
journal.replay()


The border receipt keeps a correlation identifier, remote task identifier, attempts, latency, quoted cost, and receipt identifier. It explicitly does not turn correlation metadata into permission. The replay reads the recorded tool result instead of writing the work order a second time.


### Keep production judges calibrated


In [ ]:
monitor = JudgeCalibrationMonitor(minimum_kappa=0.50, maximum_drop=0.10)
july = calibration_report(
    period="2026-07",
    judge_version="judge-v1",
    human_labels=["pass", "pass", "fail", "fail", "pass", "fail"],
    judge_labels=["pass", "pass", "fail", "fail", "pass", "pass"],
)
august = calibration_report(
    period="2026-08",
    judge_version="judge-v2",
    human_labels=["pass", "pass", "fail", "fail", "pass", "fail"],
    judge_labels=["pass", "pass", "pass", "pass", "pass", "pass"],
)
{
    "july": monitor.add(july),
    "august": monitor.add(august),
    "fresh_human_sample": stable_human_review_sample(
        [trace.trace_id for trace in traces], sample_size=5, seed=9
    ),
}


Raw agreement can remain flattering when one class dominates. The monitor watches Cohen's kappa against fresh human labels, triggers when agreement beyond chance drops, and always requires fresh calibration after a judge-version change. The sampler is seeded by the evaluation service, not by the agent, and the live agent has no grader or telemetry tool.

Cost alerting needs the same explicit coverage discipline:


In [ ]:
{
    "credit_state": credit_depletion_state(spent_usd=80, limit_usd=100),
    "coverage": detector_coverage(
        {"cloud": ["service", "account", "region"]},
        required_dimensions=["service", "account", "region", "plant", "principal"],
    ),
}


The cloud detector covers infrastructure dimensions but misses plant and principal, so the application telemetry must fill that gap. The credit policy warns at half remaining and pages at one fifth remaining rather than waiting for a delayed daily budget alert.


## 3. The real cost and the brake


![Naive and optimized cost anatomy of one turn](assets/figure_9_3_turn_cost.png)

Versioned pricing is an estimate; the invoice remains the truth. The run budget is still enforced against the estimate before every metered action, because a next-day cloud alert cannot stop today's loop.


In [ ]:
demo_budget = RunBudget(
    price_card=card,
    limit_usd=0.10,
    max_calls=4,
    max_steps=4,
    max_seconds=30,
)
args = {"asset_id": "A-PUMP-14", "fault_code": "TEMP_DRIFT"}
reservation = demo_budget.reserve_tool("check_parts_catalog", args)
demo_budget.settle(reservation, card.tool_cost("check_parts_catalog"))
try:
    demo_budget.reserve_tool("check_parts_catalog", args)
except LoopDetected as exc:
    loop_result = str(exc)
{"loop_result": loop_result, "budget": demo_budget.snapshot()}


This is a content-level refusal: the model can reuse the prior result and continue. A reservation that would cross the dollar, call, step, or wall-clock allowance raises a run-stopping error instead. Unknown models and tools are errors, never free entries.


## 4. Model routing as cost control


![Cost-quality frontier and reasoning effort shapes](assets/figure_9_4_cost_quality_frontier.png)

The useful configuration is the cheapest point above the acceptance floor, with retries and fallbacks folded into cost. The router starts from observed spans: which step and model consumed the budget?


In [ ]:
token_share_by_step(traces)


In [ ]:
candidates = load_candidates()
frontier = pareto_frontier(candidates)
selected = choose_cheapest_acceptable(candidates, quality_floor=0.90)
old = next(candidate for candidate in candidates if candidate.name == "frontier-medium")
{
    "frontier": [candidate.name for candidate in frontier],
    "selected": selected.model_dump(),
    "change_gate": routing_change_gate(old, selected, quality_floor=0.90),
}


The illustrative move from `frontier-medium` to `mid-medium` clears the quality floor and reduces cost per accepted outcome. A configuration that is merely cheaper does not ship; it must also clear the paired quality gate.


## 5. One ledger, two screens


![Finance-first value-review dashboard](assets/figure_9_5_value_dashboard.png)

Finance opens on unit economics and the discontinue list. Engineering opens on p95 steps, loops, tool failures, policy denials, model refusals, budget stops, and expensive traces. Both views drill into the same trace identifiers.


In [ ]:
finance = build_finance_dashboard(traces, quarter_budget_usd=250.0)
engineering = build_engineering_dashboard(traces)
{
    "finance_headline": finance["headline"],
    "discontinue_list": finance["discontinue_list"],
    "engineering_headline": engineering["headline"],
    "same_ledger": set(finance["drilldown_trace_ids"]) == {trace.trace_id for trace in traces},
}


### Export the value-review evidence


In [ ]:
evidence_dir = Path(tempfile.mkdtemp(prefix="chapter9-evidence-"))
manifest = emit_value_evidence(traces, evidence_dir, quarter_budget_usd=250.0)
{
    "manifest": str(manifest),
    "verified": verify_manifest(manifest),
    "artifacts": sorted(path.name for path in evidence_dir.iterdir()),
}


Open `value_review.html` for the self-contained finance snapshot. `priced_traces.jsonl` is the drilldown ledger, the two JSON files are the finance and engineering views, `token_share_by_step.json` is routing evidence, and the manifest hashes every artifact.

### Optional local observability stack

The repository includes Grafana's all-in-one development image with the OpenTelemetry Collector, Tempo, Loki, Prometheus, and Grafana. From a terminal in `chapter 9`:

```bash
docker compose up -d
python -m chapter9 --export-otlp --output-dir evidence/local
open http://localhost:3000
```

The generated finance board remains the value view. Grafana/Tempo is the trace investigation view. This cell does not start Docker or perform network I/O.


## 6. Optional Agents SDK wiring


In [ ]:
live_context = make_live_context(cases[0])
live_agent = build_maintenance_agent(live_context)
{
    "agent": live_agent.name,
    "model_adapter": type(live_agent.model).__name__,
    "tools": [tool.name for tool in live_agent.tools],
    "guardrails_per_tool": [len(tool.tool_input_guardrails) for tool in live_agent.tools],
    "has_execution_tool": any("execute" in tool.name or "approve" in tool.name for tool in live_agent.tools),
    "budget": live_context.budget.snapshot(),
}


In [ ]:
if os.getenv("OPENAI_API_KEY") and os.getenv("CHAPTER9_RUN_LIVE") == "1":
    live_result = await run_live_case(cases[0])
    live_result.model_dump()
else:
    print("Live call skipped. Set OPENAI_API_KEY and CHAPTER9_RUN_LIVE=1 to opt in.")


The live adapter reserves an illustrative model-call ceiling before every non-streamed request and uses a tool-input guardrail before every tool call. It exposes only read tools and returns a proposal; approval and execution remain outside the agent. Replace the illustrative price-card entries with your contracted rates before treating its cost as financial evidence.

## What to carry into production

- Derive tenant, plant, principal, and severity outside the model path.
- Normalize usage once and version the price card used for every estimate.
- Keep the collector, graders, and metric store outside the agent's credentials.
- Reserve budget before model and tool calls; fail closed on unknown prices.
- Count repeated successful calls as loops even when the error rate is zero.
- Replay recorded effects; rerun only an explicit fork.
- Compare fully loaded cost per accepted outcome against a measured human baseline.
- Route each step to the cheapest configuration that clears a paired quality gate.
- Put above-baseline workflows on a visible discontinue list and act on it.
